In [107]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [108]:
df=pd.read_csv("insurance_claims.csv")
df.columns

Index(['months_as_customer', 'age', 'policy_number', 'policy_bind_date',
       'policy_state', 'policy_csl', 'policy_deductable',
       'policy_annual_premium', 'umbrella_limit', 'insured_zip', 'insured_sex',
       'insured_education_level', 'insured_occupation', 'insured_hobbies',
       'insured_relationship', 'capital-gains', 'capital-loss',
       'incident_date', 'incident_type', 'collision_type', 'incident_severity',
       'authorities_contacted', 'incident_state', 'incident_city',
       'incident_location', 'incident_hour_of_the_day',
       'number_of_vehicles_involved', 'property_damage', 'bodily_injuries',
       'witnesses', 'police_report_available', 'total_claim_amount',
       'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make',
       'auto_model', 'auto_year', 'fraud_reported', '_c39'],
      dtype='object')

# Data cleaning, datatype, index 

In [109]:
# drop column as 100% nan-values
# drop umbrella limits as 80% of values are zeros, std = 1.1 Mio
df.drop(['_c39', 'umbrella_limit'],axis=1,inplace= True)

In [110]:
df.columns

Index(['months_as_customer', 'age', 'policy_number', 'policy_bind_date',
       'policy_state', 'policy_csl', 'policy_deductable',
       'policy_annual_premium', 'insured_zip', 'insured_sex',
       'insured_education_level', 'insured_occupation', 'insured_hobbies',
       'insured_relationship', 'capital-gains', 'capital-loss',
       'incident_date', 'incident_type', 'collision_type', 'incident_severity',
       'authorities_contacted', 'incident_state', 'incident_city',
       'incident_location', 'incident_hour_of_the_day',
       'number_of_vehicles_involved', 'property_damage', 'bodily_injuries',
       'witnesses', 'police_report_available', 'total_claim_amount',
       'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make',
       'auto_model', 'auto_year', 'fraud_reported'],
      dtype='object')

In [111]:
#check duplicates
df.duplicated().sum() 

np.int64(0)

In [112]:
df=df.set_index('policy_number')

In [113]:
#ensure consistency on column names
rename ={'capital-gains':'capital_gains','capital-loss': 'capital_loss'}
df.rename(columns=rename, inplace=True)

In [114]:
#change to datetime
df["policy_bind_date"]=pd.to_datetime(df['policy_bind_date'])
df["incident_date"]=pd.to_datetime(df['incident_date'])
df[['policy_bind_date', 'incident_date']].dtypes

policy_bind_date    datetime64[ns]
incident_date       datetime64[ns]
dtype: object

# Set new features

In [115]:
df['insured_zip_class']=df['insured_zip'].astype(str).str[:2]
df.drop('insured_zip', axis=1, inplace=True)

In [116]:
#additional columns year, month, date of policy bind date
df['policy_bind_year']=df["policy_bind_date"].dt.year #to be reviewed to fraud_reported
df['policy_bind_month']=df["policy_bind_date"].dt.month #to be reviewed to fraud_reported
df['policy_bind_weekday']=df["policy_bind_date"].dt.weekday+1 #to be reviewed to fraud_reportedc 

In [117]:
#additional columns month, date of incident date as year is 2015 for all cases
df['incident_month']=df["incident_date"].dt.month #to be reviewed to fraud_reported
df['incident_weekday']=df["incident_date"].dt.weekday+1 #to be reviewed to fraud_reported

In [118]:
#additional columns capital total
df['capital_total']= df['capital_gains']+df['capital_loss']

In [119]:
#Binary Encoding

#Target Encoder
df["fraud"]=df["fraud_reported"].apply(lambda x: 1 if str(x)=="Y" else 0)
df.drop("fraud_reported", axis=1, inplace=True)
#Gender Encoder
df['gender']=df['insured_sex'].apply(lambda x: 1 if str(x)=="FEMALE" else 0)
df.drop("insured_sex", axis=1, inplace=True)

In [120]:
df.to_csv('insurance_claims_clean.csv')